# ScienceQA Vision Challenge — **Training Notebook**

Fine-tunes `HuggingFaceTB/SmolVLM-500M-Instruct` on the ScienceQA visual MCQ task with **QLoRA / LoRA**, then saves a LoRA adapter that the companion `infer.ipynb` loads to generate `submission.csv`.

## Strategy (why this should beat a naive baseline)

1. **Multiple-choice log-likelihood scoring** instead of free-form generation. We score each candidate letter (`A`, `B`, …) and pick the highest. Generation with a 500M model produces verbose / wrong tokens; scoring is deterministic and a strict superset of the supervised signal.
2. **QLoRA** — base model in 4-bit NF4, trainable LoRA adapters on attention projections only. We confirm `trainable_params < 5 M` programmatically.
3. **Choice-shuffle augmentation** — at training time we randomly permute the `choices` list and re-target the answer letter. This destroys positional bias (`A` is the right answer 35 % of the time in train) and forces the model to *read* the options.
4. **Lecture/hint-aware prompting** — we use SmolVLM's chat template and inject `lecture` + `hint` only when present and short enough to fit, with a hard cap so prompts don't blow up the KV cache.
5. **Per-subject monitoring** — validation accuracy is broken down by `subject` and `grade` so you know where to spend the next training hour.
6. **bf16 mixed precision** + cosine LR + warmup + gradient accumulation. A100 likes bf16 a lot more than fp16.

## Constraints we respect

- `SmolVLM-500M-Instruct` is the only allowed checkpoint.
- ≤ 5 M trainable parameters (asserted in the LoRA cell).
- Only the provided competition data is used — no external scraped data.
- Inference runs offline (the `infer.ipynb` does not call out to the network).

> **Run order:** execute cells top-to-bottom. The training loop saves the LoRA adapter to `OUT_DIR/adapter` — point `infer.ipynb` at that path.


## 0. Install dependencies

Pinned versions matching the starter so the chat template and processor behave the same as the course baseline.

In [1]:
# Run once per Colab VM. Comment out if already installed.
%pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes==0.44.1 accelerate==1.0.1 datasets pillow matplotlib seaborn bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00


In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.6 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.44.1
    Uninstalling bitsandbytes-0.44.1:
      Successfully uninstalled bitsandbytes-0.44.1


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports & configuration

All hyper-parameters live in the `CFG` dataclass — change once, propagates everywhere.

In [4]:
import os, json, math, random, time, gc, shutil, datetime
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
SEED_E = 99
random.seed(SEED_E); np.random.seed(SEED_E); torch.manual_seed(SEED_E); torch.cuda.manual_seed_all(SEED_E)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))
    print("vram :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

for v in ("model", "optimizer", "scheduler", "trainable_params", "base_model"):
    if v in dir(): exec(f"del {v}")
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


device: cuda
gpu  : NVIDIA A100-SXM4-80GB
vram : 85.1 GB
GPU free: 84.6 GB


In [6]:
class CFG:
    # ── Paths ────────────────────────────────────────────────────────────
    data_dir: Path = Path("data")                 # contains train.csv, val.csv, test.csv, images/
    out_dir : Path = Path("outputs_train")        # adapter + plots get saved here
    adapter_dir: Path = Path("outputs_train/adapter")  # final LoRA adapter

    # ── Model ────────────────────────────────────────────────────────────
    model_id: str = "HuggingFaceTB/SmolVLM-500M-Instruct"
    img_size: int = 512                           # SmolVLM tile size; 384 keeps diagram detail
    use_4bit: bool = False                        # A100 has VRAM; bf16 LoRA trains faster & cleaner
    bf16    : bool = True

    # ── LoRA ─────────────────────────────────────────────────────────────
    lora_r       : int = 16
    lora_alpha   : int = 32
    lora_dropout : float = 0.05
    # Attention-only LoRA on the language model. Vision tower is frozen.
    lora_target_modules: tuple = ("q_proj", "k_proj", "v_proj", "o_proj")

    # ── Optim ────────────────────────────────────────────────────────────
    epochs        : int   = 8
    micro_bsz     : int   = 4         # per-step batch
    grad_accum    : int   = 4         # effective batch = 16
    eval_bsz      : int   = 8
    lr            : float = 2e-4
    weight_decay  : float = 0.0
    warmup_ratio  : float = 0.05
    max_grad_norm : float = 1.0

    # ── Prompt ───────────────────────────────────────────────────────────
    max_context_chars: int = 1200     # truncate long lectures+hints
    use_lecture: bool = True
    use_hint   : bool = True
    shuffle_choices_train: bool = True

    # ── Eval / logging ───────────────────────────────────────────────────
    log_every  : int = 25
    eval_every : int = 250            # steps; also evaluates at end of each epoch
    eval_max_samples: int = 600       # cap mid-training eval for speed; full eval at end
    save_best  : bool = True
    micro_bsz : int = 4
    grad_accum : int = 2

cfg = CFG()
cfg.adapter_dir  = Path("outputs_train/adapter_run_F")
cfg.out_dir.mkdir(parents=True, exist_ok=True)
cfg.adapter_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in cfg.__dict__.items()}, indent=2))
OCR_MAX_CHARS = 500

{
  "adapter_dir": "outputs_train/adapter_run_F"
}


## 2. Mount Drive / locate the data (Colab)

If your competition data lives in Drive, uncomment the mount cell. Otherwise edit `cfg.data_dir` to point at the unzipped folder.

In [7]:
# cfg.data_dir = Path('/content/drive/MyDrive/scienceqa')   # <-- edit me
cfg.out_dir  = Path('/content/drive/MyDrive/scienceqa/outputs_train')
cfg.adapter_dir = cfg.out_dir / 'adapter'
cfg.out_dir.mkdir(parents=True, exist_ok=True)
cfg.adapter_dir.mkdir(parents=True, exist_ok=True)

assert (cfg.data_dir / "train.csv").exists(), f"train.csv not found under {cfg.data_dir}"
assert (cfg.data_dir / "val.csv").exists(),   f"val.csv not found under {cfg.data_dir}"
assert (cfg.data_dir / "test.csv").exists(),  f"test.csv not found under {cfg.data_dir}"
print("data ok ->", cfg.data_dir.resolve())

data ok -> /content/data


In [8]:
import os

# 1. Define paths
# UPDATE THIS to match where the 'images' folder is in your Google Drive
drive_images_path = '/content/drive/MyDrive/images'

local_data_dir = '/content/data'
local_images_path = '/content/data/images'

# Ensure the local data directory exists (where the CSVs should be)
os.makedirs(local_data_dir, exist_ok=True)

# 2. Copy images to local Colab storage for faster training I/O
if not os.path.exists(local_images_path):
    print(f"Copying images from {drive_images_path} to {local_images_path}...")
    print("This might take a few minutes depending on the size, but drastically speeds up training.")
    !cp -r "{drive_images_path}" "{local_data_dir}/"
    print("Copy complete! ✅")
else:
    print("Images already exist locally! ✅")


Copying images from /content/drive/MyDrive/images to /content/data/images...
This might take a few minutes depending on the size, but drastically speeds up training.
Copy complete! ✅


## 3. Load CSVs

In [9]:
train_df = pd.read_csv(cfg.data_dir / "train.csv")
val_df   = pd.read_csv(cfg.data_dir / "val.csv")
test_df  = pd.read_csv(cfg.data_dir / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")
train_df.head(2)

train: 3,109 | val: 1,048 | test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [10]:
# OCR cache load
DRIVE_ROOT = Path("/content/drive/MyDrive/scienceqa_run_final_overnight")
RUN_E_DIR  = DRIVE_ROOT / "seed99_res512_OCR_8ep_ocr2"
RUN_E_DIR.mkdir(parents=True, exist_ok=True)



In [11]:
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, *,
                 img_size: int = 384, is_train: bool = False, shuffle_choices: bool = False):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train
        self.shuffle_choices = shuffle_choices

    def __len__(self):
        return len(self.df)

    def _load_image(self, rel_path):
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        # Long-side resize keeps aspect ratio (better for diagrams)
        w, h = img.size
        s = self.img_size / max(w, h)
        if s < 1.0:
            img = img.resize((max(1, int(w*s)), max(1, int(h*s))), Image.BICUBIC)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])
        choices = list(row["choices"])
        ans = int(row["answer"]) if "answer" in row and not pd.isna(row["answer"]) else -1

        if self.is_train and self.shuffle_choices and ans >= 0:
            perm = np.random.permutation(len(choices))
            new_choices = [choices[i] for i in perm]
            new_ans = int(np.where(perm == ans)[0][0])
            choices = new_choices
            ans = new_ans

        return {
            "id": row["id"],
            "image": img,
            "row": row,
            "choices": choices,
            "answer": ans,
        }


train_ds = ScienceQADataset(train_df, cfg.data_dir, img_size=cfg.img_size,
                            is_train=True,  shuffle_choices=cfg.shuffle_choices_train)
val_ds   = ScienceQADataset(val_df,   cfg.data_dir, img_size=cfg.img_size, is_train=False)
test_ds  = ScienceQADataset(test_df,  cfg.data_dir, img_size=cfg.img_size, is_train=False)
print("datasets:", len(train_ds), len(val_ds), len(test_ds))

datasets: 3109 1048 1008


## 5. Prompt engineering

We use SmolVLM's chat template via `processor.apply_chat_template`. The assistant turn contains *just the answer letter* — that's the only token we score / supervise.

**Why a single letter?**  Multi-token answers create label-leak risk through teacher forcing and waste capacity. The leaderboard metric is single-letter accuracy.

## 6. Dataset class

Important detail: at *train* time we optionally permute `choices`. We compute the new answer index after permutation and rebuild the prompt with the new order. The model then has to actually look at the option text to know which letter is correct.

In [12]:
# ── 3) Train + 80% val combined; 420-row monitor ────────────────────────
val_shuffled = val_df.sample(frac=1, random_state=SEED_E).reset_index(drop=True)
N_MONITOR    = 420
val_monitor  = val_shuffled.iloc[:N_MONITOR].reset_index(drop=True)
val_train    = val_shuffled.iloc[N_MONITOR:].reset_index(drop=True)
combined_train = pd.concat([train_df, val_train], ignore_index=True).sample(frac=1, random_state=SEED_E).reset_index(drop=True)
print(f"Combined train: {len(combined_train):,} | Monitor: {len(val_monitor):,}")


Combined train: 3,737 | Monitor: 420


In [13]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def _truncate(s: str, n: int) -> str:
    s = str(s).strip()
    return s if len(s) <= n else s[: n - 1] + "…"

def build_user_text(row, max_ctx: int = 1200, use_lecture=True, use_hint=True,
                    choices: Optional[List[str]] = None) -> str:
    """User-turn text. Image is attached separately by the processor."""
    if choices is None:
        choices = row["choices"]

    parts = []
    if use_lecture:
        lec = row.get("lecture", None)
        if isinstance(lec, str) and lec.strip():
            parts.append(_truncate(lec, max_ctx))
    if use_hint:
        hint = row.get("hint", None)
        if isinstance(hint, str) and hint.strip():
            parts.append(_truncate(hint, max_ctx // 2))
    context = "\n".join(parts)

    choice_lines = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    text  = ""
    if context:
        text += f"Context:\n{context}\n\n"
    text += f"Question: {row['question']}\n\n"
    text += f"Choices:\n{choice_lines}\n\n"
    text += "Answer with a single letter only."
    return text


def build_messages(row, *, include_answer: bool, choices=None, answer_idx=None):
    user_text = build_user_text(
        row,
        max_ctx=cfg.max_context_chars,
        use_lecture=cfg.use_lecture,
        use_hint=cfg.use_hint,
        choices=choices,
    )
    msgs = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_text},
        ]}
    ]
    if include_answer:
        idx = answer_idx if answer_idx is not None else int(row["answer"])
        msgs.append({"role": "assistant", "content": [
            {"type": "text", "text": CHOICE_LETTERS[idx]},
        ]})
    return msgs


# Quick sanity print
sample = train_df.iloc[0]
for m in build_messages(sample, include_answer=True):
    print(m["role"].upper(), "→", m["content"][-1]["text"][:200], "..." if len(m["content"][-1]["text"]) > 200 else "")

USER → Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get p ...
ASSISTANT → C 


In [14]:
# ── 4) OCR-aware prompt builder ──────────────────────────────────────────


def build_user_text_ocr(row, *, max_ctx=cfg.max_context_chars,
                         use_lecture=True, use_hint=True, choices=None):
    if choices is None: choices = row["choices"]
    parts = []
    if use_lecture:
        lec = row.get("lecture", None)
        if isinstance(lec, str) and lec.strip(): parts.append(_truncate(lec, max_ctx))
    if use_hint:
        hint = row.get("hint", None)
        if isinstance(hint, str) and hint.strip(): parts.append(_truncate(hint, max_ctx//2))
    # NEW: OCR text from the image
    ocr = OCR_TEXT.get(row["id"], "") or ""
    ocr = ocr.strip()
    if ocr:
        parts.append(f"Text extracted from image (via OCR): {ocr[:OCR_MAX_CHARS]}")
    context = "\n".join(parts)
    choice_lines = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    text = ""
    if context: text += f"Context:\n{context}\n\n"
    text += f"Question: {row['question']}\n\nChoices:\n{choice_lines}\n\nAnswer with a single letter only."
    return text

def build_messages_ocr(row, *, include_answer, choices=None, answer_idx=None):
    user_text = build_user_text_ocr(
        row, max_ctx=cfg.max_context_chars,
        use_lecture=cfg.use_lecture, use_hint=cfg.use_hint, choices=choices,
    )
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":user_text}]}]
    if include_answer:
        idx = answer_idx if answer_idx is not None else int(row["answer"])
        msgs.append({"role":"assistant","content":[{"type":"text","text":CHOICE_LETTERS[idx]}]})
    return msgs

# ── 5) OCR-aware collator + scoring (overrides for this run) ────────────
def make_train_batch_ocr(items):
    full_texts, prompt_texts, images = [], [], []
    for it in items:
        mf = build_messages_ocr(it["row"], include_answer=True,
                                 choices=it["choices"], answer_idx=it["answer"])
        mp = build_messages_ocr(it["row"], include_answer=False, choices=it["choices"])
        full_texts.append(processor.apply_chat_template(mf, add_generation_prompt=False))
        prompt_texts.append(processor.apply_chat_template(mp, add_generation_prompt=True))
        images.append([it["image"]])
    processor.tokenizer.padding_side = "right"
    enc = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    plens = []
    for p, im in zip(prompt_texts, images):
        pe = processor(text=p, images=im, return_tensors="pt")
        plens.append(int(pe["input_ids"].shape[1]))
    labels = enc["input_ids"].clone()
    for i, pl in enumerate(plens): labels[i, :pl] = -100
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    processor.tokenizer.padding_side = "left"
    return enc

@torch.inference_mode()
def score_batch_ocr(model_, items):
    prompts, images, ncs = [], [], []
    for it in items:
        msgs = build_messages_ocr(it["row"], include_answer=False, choices=it["choices"])
        prompts.append(processor.apply_chat_template(msgs, add_generation_prompt=True))
        images.append([it["image"]]); ncs.append(len(it["choices"]))
    processor.tokenizer.padding_side = "left"
    inp = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inp = {k:(v.to(model_.device) if torch.is_tensor(v) else v) for k,v in inp.items()}
    out = model_(**inp); last = out.logits[:, -1, :]
    preds = []
    for i, n in enumerate(ncs):
        ids = [LETTER_IDS[CHOICE_LETTERS[k]] for k in range(n)]
        preds.append(int(last[i, ids].argmax().item()))
    return np.array(preds)

def evaluate_ocr(model_, dataset, max_samples=None, bs=16):
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    correct = total = 0
    model_.eval()
    for s in range(0, n, bs):
        items = [dataset[s+j] for j in range(min(bs, n-s))]
        gts = np.array([it["answer"] for it in items])
        correct += int((score_batch_ocr(model_, items) == gts).sum()); total += len(items)
    return {"accuracy": correct/max(1,total), "n": total}

In [15]:
# ── 6) Datasets / loader ─────────────────────────────────────────────────
train_ds = ScienceQADataset(combined_train, cfg.data_dir, img_size=cfg.img_size,
                            is_train=True, shuffle_choices=True)
val_ds   = ScienceQADataset(val_monitor,   cfg.data_dir, img_size=cfg.img_size, is_train=False)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=cfg.micro_bsz, shuffle=True,
    collate_fn=make_train_batch_ocr, num_workers=0, pin_memory=True, drop_last=True,
)
print(f"steps/epoch={len(train_loader)}  effective_batch={cfg.micro_bsz*cfg.grad_accum}")


steps/epoch=934  effective_batch=8


In [16]:
# ── 7) Fresh base + fresh LoRA ───────────────────────────────────────────
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForImageTextToText, get_cosine_schedule_with_warmup

processor = AutoProcessor.from_pretrained(cfg.model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
# left-padding is essential for causal-LM scoring (we read the LAST non-pad logit)
processor.tokenizer.padding_side = "left"

model_kwargs: Dict[str, Any] = dict(low_cpu_mem_usage=True)
if cfg.use_4bit:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["dtype"] = torch.bfloat16 if cfg.bf16 else torch.float32
    model_kwargs["device_map"] = "auto" if torch.cuda.is_available() else None

from peft import LoraConfig, get_peft_model
base_model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id, dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
)
for p in base_model.parameters(): p.requires_grad = False
base_model.config.use_cache = False
lora_cfg_E = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(base_model, lora_cfg_E)
# 80GB A100 → leave grad checkpointing OFF for ~30% speedup
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}"); assert trainable <= 5_000_000


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable params: 4,784,128


In [17]:
# ── 8) Optim + cosine sized for full N epochs ────────────────────────────
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_update_steps = (len(train_loader) // cfg.grad_accum) * cfg.epochs
warmup_steps = max(1, int(total_update_steps * cfg.warmup_ratio))
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
print(f"total_updates={total_update_steps}  warmup={warmup_steps}  lr_peak={cfg.lr}")


total_updates=3736  warmup=186  lr_peak=0.0002


In [18]:
# ── 9) Save-best mirror ──────────────────────────────────────────────────
def _save_best_e(acc, step):
    global best_acc, best_step
    if acc > best_acc:
        best_acc, best_step = acc, step
        model.save_pretrained(cfg.adapter_dir)
        processor.save_pretrained(cfg.adapter_dir)
        for f in Path(cfg.adapter_dir).glob("*"):
            if f.is_file(): shutil.copy2(f, RUN_E_DIR / f.name)
        print(f"  ★ new best ({acc*100:.2f}%) — mirrored to Drive: {RUN_E_DIR}")


In [19]:
# Determine the token ID for each letter.
# We tokenize the letter the way the assistant would emit it (no leading space,
# the chat template already places us right at the assistant prefix).
def _letter_token_id(letter: str) -> int:
    ids = processor.tokenizer(letter, add_special_tokens=False).input_ids
    if len(ids) != 1:
        # fallback: try the variant used after assistant prefix
        ids = processor.tokenizer(" " + letter, add_special_tokens=False).input_ids
    assert len(ids) >= 1, f"could not tokenize letter {letter!r}"
    return ids[-1]

LETTER_IDS = {L: _letter_token_id(L) for L in CHOICE_LETTERS}
print({L: LETTER_IDS[L] for L in CHOICE_LETTERS})


@torch.inference_mode()
def score_batch(model, batch_items, *, choices_override: Optional[List[List[str]]] = None) -> np.ndarray:
    """Return predicted index per item via log-likelihood scoring of the answer letter."""
    images, prompts, num_choices = [], [], []
    for i, item in enumerate(batch_items):
        choices = choices_override[i] if choices_override is not None else item["choices"]
        msgs = build_messages(item["row"], include_answer=False, choices=choices)
        prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
        prompts.append(prompt)
        images.append([item["image"]])
        num_choices.append(len(choices))

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

    out = model(**inputs)
    logits = out.logits  # (B, T, V)
    last = logits[:, -1, :]  # (B, V); left-padded so position -1 is the next-token slot
    preds = []
    for i, n in enumerate(num_choices):
        cand_ids = [LETTER_IDS[CHOICE_LETTERS[k]] for k in range(n)]
        cand_logits = last[i, cand_ids]
        preds.append(int(cand_logits.argmax().item()))
    return np.array(preds, dtype=np.int64)


def evaluate(model, dataset, *, max_samples: Optional[int] = None,
             return_per_example: bool = False, batch_size: Optional[int] = None) -> Dict[str, Any]:
    bs = batch_size or cfg.eval_bsz
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    correct, total = 0, 0
    preds_all, gts_all, ids_all = [], [], []
    model.eval()
    for start in range(0, n, bs):
        items = [dataset[start + j] for j in range(min(bs, n - start))]
        gts = np.array([it["answer"] for it in items])
        preds = score_batch(model, items)
        correct += int((preds == gts).sum())
        total   += len(items)
        if return_per_example:
            preds_all.extend(preds.tolist())
            gts_all.extend(gts.tolist())
            ids_all.extend([it["id"] for it in items])
    acc = correct / max(1, total)
    out = {"accuracy": acc, "n": total}
    if return_per_example:
        out["preds"] = preds_all; out["gts"] = gts_all; out["ids"] = ids_all
    return out



{'A': 49, 'B': 50, 'C': 51, 'D': 52, 'E': 53, 'F': 54, 'G': 55, 'H': 56, 'I': 57, 'J': 58}


In [20]:
def make_train_batch(items):
    """Build a model-ready training batch from raw dataset items.

    Subtlety: image-token expansion happens inside the *processor*, not the tokenizer.
    To get the right boundary between prompt and answer in input_ids, we re-encode
    the prompt-only version through the processor (with the same image) and read its
    length — that includes the expanded image tokens. The text tokenizer alone would
    undercount and we'd end up putting `-100` over the answer letter (loss = NaN).
    """
    images, full_texts, prompt_texts = [], [], []
    for it in items:
        msgs_full   = build_messages(it["row"], include_answer=True,
                                     choices=it["choices"], answer_idx=it["answer"])
        msgs_prompt = build_messages(it["row"], include_answer=False, choices=it["choices"])
        full   = processor.apply_chat_template(msgs_full,   add_generation_prompt=False)
        prompt = processor.apply_chat_template(msgs_prompt, add_generation_prompt=True)
        full_texts.append(full)
        prompt_texts.append(prompt)
        images.append([it["image"]])

    # Right-padding for training: prompt tokens sit at positions 0..plen, answer
    # at plen, then end-of-utterance, then padding.
    processor.tokenizer.padding_side = "right"
    enc = processor(text=full_texts, images=images, return_tensors="pt", padding=True)

    # Per-row prompt length WITH image-token expansion. Encode each separately
    # (no padding) and read input_ids.shape[1].
    prompt_lens = []
    for p, im in zip(prompt_texts, images):
        pe = processor(text=p, images=im, return_tensors="pt")
        prompt_lens.append(int(pe["input_ids"].shape[1]))

    labels = enc["input_ids"].clone()
    for i, plen in enumerate(prompt_lens):
        labels[i, :plen] = -100          # mask the prompt
    labels[enc["attention_mask"] == 0] = -100   # mask padding
    enc["labels"] = labels

    # leave padding_side = "right" here; the next score_batch call resets it via the
    # LEFT-padding setup it relies on. We reset it explicitly to be safe:
    processor.tokenizer.padding_side = "left"
    return enc


def train_collate(items):
    return make_train_batch(items)


# Quick correctness check on a tiny batch BEFORE we start the full loop
sample_batch_items = [train_ds[i] for i in range(min(2, len(train_ds)))]
_chk = make_train_batch(sample_batch_items)
_lab = _chk["labels"]
_kept = (_lab != -100).sum().item()
print(f"sanity: labels kept (non -100) on 2-row batch = {_kept}  (should be ~2-4 tokens, one per row)")
assert _kept >= 1, "label masking too aggressive — every position is -100"
del _chk, sample_batch_items, _lab, _kept

# IMPORTANT: num_workers=0 keeps the processor side-effects (padding_side flips) on
# the main process so they don't get out of sync with score_batch.
train_loader = DataLoader(
    train_ds, batch_size=cfg.micro_bsz, shuffle=True,
    collate_fn=train_collate, num_workers=0, pin_memory=True, drop_last=True,
)
print("steps per epoch:", len(train_loader), "| effective batch:", cfg.micro_bsz * cfg.grad_accum)

sanity: labels kept (non -100) on 2-row batch = 6  (should be ~2-4 tokens, one per row)
steps per epoch: 934 | effective batch: 8


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# PADDLEOCR PREPROCESSING — extracts text with better quality than EasyOCR.
# Especially helpful for Punnett squares, tables, and labeled diagrams.
# Runs on GPU. Resumable — re-running skips already-processed ids.
# Expected runtime: ~50-80 min on A100 80GB for ~10k images.
# Saves: ocr_text_paddle.json (separate from the EasyOCR cache so both exist)
# ═══════════════════════════════════════════════════════════════════════════
import os, json, time, gc
from pathlib import Path
import pandas as pd
import torch

# ── Drive ─────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError: pass

DATA_DIR  = Path("/content/data")
DRIVE_OUT = Path("/content/drive/MyDrive/scienceqa_run_final_overnight")
OCR_JSON  = DRIVE_OUT / "ocr_text_paddle.json"   # ← separate file
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
SAVE_EVERY = 200

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:


# ── Resume support ───────────────────────────────────────────────────────
ocr_by_id = {}
if OCR_JSON.exists():
    ocr_by_id = json.loads(OCR_JSON.read_text())
    print(f"resuming — {len(ocr_by_id):,} already done")

def _load(name):
    df = pd.read_csv(DATA_DIR / f"{name}.csv")
    return df[["id", "image_path"]]
all_rows = pd.concat([_load("train"), _load("val"), _load("test")], ignore_index=True).drop_duplicates("id").reset_index(drop=True)
todo = all_rows[~all_rows["id"].isin(ocr_by_id)].reset_index(drop=True)
print(f"{len(all_rows):,} total | {len(todo):,} to process")


# Replace the entire install + init section with:
get_ipython().system("pip install -q python-doctr[torch]")

from doctr.io import DocumentFile
from doctr.models import ocr_predictor
ocr = ocr_predictor(pretrained=True).cuda()
print("DocTR ready, GPU =", torch.cuda.is_available())

def extract(path):
    try:
        doc = DocumentFile.from_images(str(path))
        result = ocr(doc)
        # Concatenate all words across the page
        words = []
        for page in result.pages:
            for block in page.blocks:
                for line in block.lines:
                    for word in line.words:
                        if word.confidence >= 0.4:
                            words.append(word.value)
        return " ".join(words)[:600]
    except Exception:
        return ""

# ── Process loop ─────────────────────────────────────────────────────────
t0 = time.time()
for i, row in todo.iterrows():
    p = DATA_DIR / row["image_path"]
    if not p.exists():
        ocr_by_id[row["id"]] = ""; continue
    ocr_by_id[row["id"]] = extract(p)
    if (i + 1) % SAVE_EVERY == 0:
        OCR_JSON.write_text(json.dumps(ocr_by_id))
        rate = (i+1) / (time.time() - t0)
        eta_min = (len(todo) - (i+1)) / rate / 60
        print(f"  {i+1:>5d}/{len(todo)}  rate={rate:.1f} img/s  ETA={eta_min:.1f} min  "
              f"sample={ocr_by_id[row['id']][:80]!r}")

OCR_JSON.write_text(json.dumps(ocr_by_id))

# ── Sanity ───────────────────────────────────────────────────────────────
n_with = sum(1 for v in ocr_by_id.values() if v.strip())
print(f"\n✅ done. {n_with:,}/{len(ocr_by_id):,} have text ({n_with/len(ocr_by_id)*100:.1f}%)  in {(time.time()-t0)/60:.1f} min")

# Compare a few against the EasyOCR cache so you can see the quality diff
import random
ez_path = DRIVE_OUT / "ocr_text.json"
if ez_path.exists():
    ez = json.loads(ez_path.read_text())
    print("\n— Quality check vs. EasyOCR (5 random samples) —")
    random.seed(0)
    for sid in random.sample(list(ocr_by_id.keys()), 5):
        print(f"\n{sid}")
        print(f"  EasyOCR : {ez.get(sid, '')[:200]!r}")
        print(f"  Paddle  : {ocr_by_id[sid][:200]!r}")

# NOTE: removed del ocr to allow re-running this cell safely.
gc.collect(); torch.cuda.empty_cache()
print("\nGPU cleared. Now run mega_run_G_cell.py for training.")

5,165 total | 5,165 to process
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 43.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 31.0 MB/s eta 0:00:00


  0%|          | 0/65814772 [00:00<?, ?it/s]

  0%|          | 0/63303144 [00:00<?, ?it/s]

DocTR ready, GPU = True
    200/5165  rate=15.7 img/s  ETA=5.3 min  sample='E -'
    400/5165  rate=17.2 img/s  ETA=4.6 min  sample="Danielle's lunch Akira's lunch mn - - a"
    600/5165  rate=17.8 img/s  ETA=4.3 min  sample='Pair 1 Pair 2 > N > N S N S N I 29 mm I 29 mm -'
    800/5165  rate=17.0 img/s  ETA=4.3 min  sample='Pair 2 Pair 1 N - > > - > > 1.5 in - 1.5 in'
   1000/5165  rate=16.3 img/s  ETA=4.2 min  sample='Pair 1 Pair 2 S N N S N S S N 5 mm 3 mm'
   1200/5165  rate=16.2 img/s  ETA=4.1 min  sample='Items Jen wants Items Nate wants a sandwich a hot dog oranges tomatoes broccoli '
   1400/5165  rate=16.7 img/s  ETA=3.8 min  sample=''
   1600/5165  rate=17.2 img/s  ETA=3.4 min  sample=''
   1800/5165  rate=17.3 img/s  ETA=3.2 min  sample='persimmon tree bobcat black bear gray fox black racer bolete beaver fungus swall'
   2000/5165  rate=16.7 img/s  ETA=3.2 min  sample='- - V I , / ff Sample A Sample B Mass of each particle: 28 u Mass of each partic'
   2200/5165  rate=16.0 i

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN G — same recipe as Run E (the 0.90342 winner) but uses the PADDLEOCR
# cache instead of EasyOCR. This is a controlled experiment: if Run G beats E
# individually on the LB, PaddleOCR was the difference.
#
# Prereqs: Phase 1 cells already executed (cfg, ScienceQADataset, etc. in scope),
# AND paddleocr_preprocess_cell.py has been run (ocr_text_paddle.json on Drive).
# ═══════════════════════════════════════════════════════════════════════════
import os, gc, json, time, random, shutil, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW

# Free leftover GPU memory
for v in ("model", "optimizer", "scheduler", "trainable_params", "base_model"):
    if v in dir(): exec(f"del {v}")
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError: pass

# ── Paths ────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path("/content/drive/MyDrive/scienceqa_run_final_overnight")
OCR_JSON   = DRIVE_ROOT / "ocr_text_paddle.json"           # ← NEW (PaddleOCR)
RUN_G_DIR  = DRIVE_ROOT / "seed99_res512_PaddleOCR_8ep"    # ← NEW output dir
RUN_G_DIR.mkdir(parents=True, exist_ok=True)
assert OCR_JSON.exists(), f"missing {OCR_JSON} — run paddleocr_preprocess_cell.py first"

OCR_TEXT = json.loads(OCR_JSON.read_text())
n_with = sum(1 for v in OCR_TEXT.values() if v.strip())
print(f"PaddleOCR cache: {n_with:,}/{len(OCR_TEXT):,} images have text")

# ── Settings (mirror Run E exactly except OCR source) ────────────────────
SEED_G = 99               # SAME as E so we isolate "PaddleOCR vs EasyOCR" as the only diff
random.seed(SEED_G); np.random.seed(SEED_G); torch.manual_seed(SEED_G); torch.cuda.manual_seed_all(SEED_G)

cfg.epochs       = 8
cfg.img_size     = 512
cfg.lr           = 2e-4
cfg.micro_bsz    = 4
cfg.grad_accum   = 4
cfg.adapter_dir  = Path("outputs_train/adapter_run_G")
cfg.adapter_dir.mkdir(parents=True, exist_ok=True)

OCR_MAX_CHARS = 1000       # match Run E's training distribution

# ── Train + 80% val combined; 420-row monitor ───────────────────────────
val_shuffled = val_df.sample(frac=1, random_state=SEED_G).reset_index(drop=True)
N_MONITOR    = 420
val_monitor  = val_shuffled.iloc[:N_MONITOR].reset_index(drop=True)
val_train    = val_shuffled.iloc[N_MONITOR:].reset_index(drop=True)
combined_train = pd.concat([train_df, val_train], ignore_index=True).sample(frac=1, random_state=SEED_G).reset_index(drop=True)
print(f"Combined train: {len(combined_train):,} | Monitor: {len(val_monitor):,}")

# ── Prompt builder using the PADDLE OCR cache ────────────────────────────
def build_user_text_paddle(row, *, max_ctx=cfg.max_context_chars,
                            use_lecture=True, use_hint=True, choices=None):
    if choices is None: choices = row["choices"]
    parts = []
    if use_lecture:
        lec = row.get("lecture", None)
        if isinstance(lec, str) and lec.strip(): parts.append(_truncate(lec, max_ctx))
    if use_hint:
        hint = row.get("hint", None)
        if isinstance(hint, str) and hint.strip(): parts.append(_truncate(hint, max_ctx//2))
    ocr = (OCR_TEXT.get(row["id"], "") or "").strip()
    if ocr:
        parts.append(f"Text extracted from image (via OCR): {ocr[:OCR_MAX_CHARS]}")
    context = "\n".join(parts)
    cl = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    text = ""
    if context: text += f"Context:\n{context}\n\n"
    text += f"Question: {row['question']}\n\nChoices:\n{cl}\n\nAnswer with a single letter only."
    return text

def build_messages_paddle(row, *, include_answer, choices=None, answer_idx=None):
    user_text = build_user_text_paddle(
        row, max_ctx=cfg.max_context_chars,
        use_lecture=cfg.use_lecture, use_hint=cfg.use_hint, choices=choices,
    )
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":user_text}]}]
    if include_answer:
        idx = answer_idx if answer_idx is not None else int(row["answer"])
        msgs.append({"role":"assistant","content":[{"type":"text","text":CHOICE_LETTERS[idx]}]})
    return msgs

# ── Collator + scoring (Paddle-OCR aware) ────────────────────────────────
def make_train_batch_paddle(items):
    full_texts, prompt_texts, images = [], [], []
    for it in items:
        mf = build_messages_paddle(it["row"], include_answer=True,
                                    choices=it["choices"], answer_idx=it["answer"])
        mp = build_messages_paddle(it["row"], include_answer=False, choices=it["choices"])
        full_texts.append(processor.apply_chat_template(mf, add_generation_prompt=False))
        prompt_texts.append(processor.apply_chat_template(mp, add_generation_prompt=True))
        images.append([it["image"]])
    processor.tokenizer.padding_side = "right"
    enc = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    plens = []
    for p, im in zip(prompt_texts, images):
        pe = processor(text=p, images=im, return_tensors="pt")
        plens.append(int(pe["input_ids"].shape[1]))
    labels = enc["input_ids"].clone()
    for i, pl in enumerate(plens): labels[i, :pl] = -100
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    processor.tokenizer.padding_side = "left"
    return enc

@torch.inference_mode()
def score_batch_paddle(model_, items):
    prompts, images, ncs = [], [], []
    for it in items:
        msgs = build_messages_paddle(it["row"], include_answer=False, choices=it["choices"])
        prompts.append(processor.apply_chat_template(msgs, add_generation_prompt=True))
        images.append([it["image"]]); ncs.append(len(it["choices"]))
    processor.tokenizer.padding_side = "left"
    inp = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inp = {k:(v.to(model_.device) if torch.is_tensor(v) else v) for k,v in inp.items()}
    out = model_(**inp); last = out.logits[:, -1, :]
    preds = []
    for i, n in enumerate(ncs):
        ids = [LETTER_IDS[CHOICE_LETTERS[k]] for k in range(n)]
        preds.append(int(last[i, ids].argmax().item()))
    return np.array(preds)

def evaluate_paddle(model_, dataset, max_samples=None, bs=16):
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    correct = total = 0
    model_.eval()
    for s in range(0, n, bs):
        items = [dataset[s+j] for j in range(min(bs, n-s))]
        gts = np.array([it["answer"] for it in items])
        correct += int((score_batch_paddle(model_, items) == gts).sum()); total += len(items)
    return {"accuracy": correct/max(1,total), "n": total}

# ── Datasets / loader ────────────────────────────────────────────────────
train_ds = ScienceQADataset(combined_train, cfg.data_dir, img_size=cfg.img_size,
                             is_train=True, shuffle_choices=True)
val_ds   = ScienceQADataset(val_monitor,   cfg.data_dir, img_size=cfg.img_size, is_train=False)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=cfg.micro_bsz, shuffle=True,
    collate_fn=make_train_batch_paddle, num_workers=0, pin_memory=True, drop_last=True,
)
print(f"steps/epoch={len(train_loader)}  effective_batch={cfg.micro_bsz*cfg.grad_accum}")

# ── Fresh base + fresh LoRA (same topology as Run E) ────────────────────
from transformers import AutoModelForImageTextToText, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
base_model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id, dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
)
for p in base_model.parameters(): p.requires_grad = False
base_model.config.use_cache = False
lora_cfg_G = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(base_model, lora_cfg_G)
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"): model.enable_input_require_grads()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}"); assert trainable <= 5_000_000

# ── Optim + cosine ───────────────────────────────────────────────────────
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_update_steps = (len(train_loader) // cfg.grad_accum) * cfg.epochs
warmup_steps = max(1, int(total_update_steps * cfg.warmup_ratio))
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
print(f"total_updates={total_update_steps}  warmup={warmup_steps}  lr_peak={cfg.lr}")

# ── Save-best mirror ─────────────────────────────────────────────────────
def _save_best_g(acc, step):
    global best_acc, best_step
    if acc > best_acc:
        best_acc, best_step = acc, step
        model.save_pretrained(cfg.adapter_dir)
        processor.save_pretrained(cfg.adapter_dir)
        for f in Path(cfg.adapter_dir).glob("*"):
            if f.is_file(): shutil.copy2(f, RUN_G_DIR / f.name)
        print(f"  ★ new best ({acc*100:.2f}%) — mirrored to Drive: {RUN_G_DIR}")

# ── Train ────────────────────────────────────────────────────────────────
history = {"step":[], "loss":[], "eval_step":[], "eval_acc":[]}
best_acc, best_step = -1.0, -1
update_step = micro_step = 0
running_loss = 0.0; loss_count = 0
t0 = time.time()

for epoch in range(cfg.epochs):
    print(f"\n=== Run G (PaddleOCR)  Epoch {epoch+1}/{cfg.epochs} ===")
    model.train(); optimizer.zero_grad(set_to_none=True)
    for batch in train_loader:
        batch = {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in batch.items()}
        out = model(**batch)
        loss = out.loss / cfg.grad_accum
        loss.backward()
        running_loss += loss.item() * cfg.grad_accum; loss_count += 1
        micro_step += 1
        if micro_step % cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
            update_step += 1
            if update_step % cfg.log_every == 0:
                avg = running_loss / max(1, loss_count); running_loss = 0.0; loss_count = 0
                history["step"].append(update_step); history["loss"].append(avg)
                print(f"  step {update_step:>5d}/{total_update_steps}  loss={avg:.4f}  "
                      f"lr={scheduler.get_last_lr()[0]:.2e}  elapsed={(time.time()-t0)/60:.1f}m")
            if update_step % cfg.eval_every == 0:
                ev = evaluate_paddle(model, val_ds)
                history["eval_step"].append(update_step); history["eval_acc"].append(ev["accuracy"])
                print(f"  ↪ monitor acc (n={ev['n']}): {ev['accuracy']*100:.2f}%")
                _save_best_g(ev["accuracy"], update_step); model.train()
    ev = evaluate_paddle(model, val_ds)
    history["eval_step"].append(update_step); history["eval_acc"].append(ev["accuracy"])
    print(f"--- end epoch {epoch+1} monitor acc (n={ev['n']}): {ev['accuracy']*100:.2f}%")
    _save_best_g(ev["accuracy"], update_step)

print(f"\n✅ Run G done in {(time.time()-t0)/60:.1f} min — best monitor = {best_acc*100:.2f}%")

# ── Save meta ────────────────────────────────────────────────────────────
(RUN_G_DIR / "meta.json").write_text(json.dumps({
    "run":"G_PaddleOCR_8ep",
    "seed": SEED_G, "img_size": cfg.img_size, "epochs": cfg.epochs, "lr": cfg.lr,
    "best_monitor_acc": float(best_acc), "best_step": int(best_step),
    "trainable_params": int(trainable),
    "elapsed_min": round((time.time()-t0)/60, 1),
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "training_data": {"n_train_orig": int(len(train_df)),
                       "n_val_added": int(len(val_train)),
                       "monitor_n": int(len(val_monitor))},
    "ocr_source": "PaddleOCR", "ocr_max_chars": OCR_MAX_CHARS,
}, indent=2))
(RUN_G_DIR / "history.json").write_text(json.dumps(history))

# ── Disconnect ───────────────────────────────────────────────────────────
print("Sleeping 60s for Drive flush…")
time.sleep(60)
try:
    from google.colab import runtime; runtime.unassign()
except Exception as e:
    print(e); os._exit(0)


In [ ]:
# # ── 10) Train ────────────────────────────────────────────────────────────
# history = {"step":[], "loss":[], "eval_step":[], "eval_acc":[]}
# best_acc, best_step = -1.0, -1
# update_step = micro_step = 0
# running_loss = 0.0; loss_count = 0
# t0 = time.time()

# for epoch in range(cfg.epochs):
#     print(f"\n=== Run E (mega)  Epoch {epoch+1}/{cfg.epochs} ===")
#     model.train(); optimizer.zero_grad(set_to_none=True)
#     for batch in train_loader:
#         batch = {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in batch.items()}
#         out = model(**batch)
#         loss = out.loss / cfg.grad_accum
#         loss.backward()
#         running_loss += loss.item() * cfg.grad_accum; loss_count += 1
#         micro_step += 1
#         if micro_step % cfg.grad_accum == 0:
#             torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
#             optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
#             update_step += 1
#             if update_step % cfg.log_every == 0:
#                 avg = running_loss / max(1, loss_count); running_loss = 0.0; loss_count = 0
#                 history["step"].append(update_step); history["loss"].append(avg)
#                 print(f"  step {update_step:>5d}/{total_update_steps}  loss={avg:.4f}  "
#                       f"lr={scheduler.get_last_lr()[0]:.2e}  elapsed={(time.time()-t0)/60:.1f}m")
#             if update_step % cfg.eval_every == 0:
#                 ev = evaluate_ocr(model, val_ds)
#                 history["eval_step"].append(update_step); history["eval_acc"].append(ev["accuracy"])
#                 print(f"  ↪ monitor acc (n={ev['n']}): {ev['accuracy']*100:.2f}%")
#                 _save_best_e(ev["accuracy"], update_step); model.train()
#     ev = evaluate_ocr(model, val_ds)
#     history["eval_step"].append(update_step); history["eval_acc"].append(ev["accuracy"])
#     print(f"--- end epoch {epoch+1} monitor acc (n={ev['n']}): {ev['accuracy']*100:.2f}%")
#     _save_best_e(ev["accuracy"], update_step)

# print(f"\n✅ Run E done in {(time.time()-t0)/60:.1f} min — best monitor = {best_acc*100:.2f}%")



=== Run E (mega)  Epoch 1/8 ===
  step    25/3736  loss=9.4570  lr=2.69e-05  elapsed=3.4m
  step    50/3736  loss=5.0405  lr=5.38e-05  elapsed=6.7m
  step    75/3736  loss=0.3325  lr=8.06e-05  elapsed=10.1m
  step   100/3736  loss=0.2227  lr=1.08e-04  elapsed=13.4m
  step   125/3736  loss=0.1954  lr=1.34e-04  elapsed=16.7m
  step   150/3736  loss=0.2028  lr=1.61e-04  elapsed=20.0m
  step   175/3736  loss=0.1906  lr=1.88e-04  elapsed=23.3m
  step   200/3736  loss=0.1860  lr=2.00e-04  elapsed=26.5m
  step   225/3736  loss=0.1836  lr=2.00e-04  elapsed=29.8m
  step   250/3736  loss=0.1806  lr=2.00e-04  elapsed=33.2m
  ↪ monitor acc (n=420): 66.67%
  ★ new best (66.67%) — mirrored to Drive: /content/drive/MyDrive/scienceqa_run_final_overnight/seed11_res512_OCR_8ep
  step   275/3736  loss=0.2028  lr=2.00e-04  elapsed=40.1m
  step   300/3736  loss=0.1760  lr=1.99e-04  elapsed=43.5m
  step   325/3736  loss=0.1652  lr=1.99e-04  elapsed=46.8m
  step   350/3736  loss=0.1597  lr=1.99e-04  elapsed

In [ ]:
# ── 11) Save meta ────────────────────────────────────────────────────────
(RUN_E_DIR / "meta.json").write_text(json.dumps({
    "run":"F_mega_OCR_8ep",
    "seed": SEED_E, "img_size": cfg.img_size, "epochs": cfg.epochs, "lr": cfg.lr,
    "best_monitor_acc": float(best_acc), "best_step": int(best_step),
    "trainable_params": int(trainable),
    "elapsed_min": round((time.time()-t0)/60, 1),
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "training_data": {"n_train_orig": int(len(train_df)),
                       "n_val_added": int(len(val_train)),
                       "monitor_n": int(len(val_monitor))},
    "ocr_used": True, "ocr_max_chars": OCR_MAX_CHARS,
    "ocr_pct_with_text": round(n_with/len(OCR_TEXT)*100, 1),
}, indent=2))
(RUN_E_DIR / "history.json").write_text(json.dumps(history))


4623

## 13. Plot training curves

In [ ]:
# ── 10) Disconnect to stop credit burn ────────────────────────────────────
print("\nSleeping 60s to flush Drive…")
time.sleep(60)
print("Disconnecting runtime. 💤")
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("runtime.unassign failed:", e)
    os._exit(0)



Sleeping 60s to flush Drive…
Disconnecting runtime. 💤
